# Solar Filament Segmentation - Colab Training

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# create project folder in Drive
project_drive = '/content/drive/MyDrive/filament_kaggle'
os.makedirs(project_drive, exist_ok=True)

!pip install -q segmentation-models-pytorch==0.3.3 albumentations==2.0.8 kaggle kagglehub


In [ ]:
import os, kagglehub, shutil

# set your Kaggle API access token from userdata, or paste it below
from google.colab import userdata
try:
    os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
except Exception as e:
    # fallback: paste token here manually
    os.environ['KAGGLE_API_TOKEN'] = 'YOUR_TOKEN_HERE'

# download competition data via kagglehub
path = kagglehub.competition_download('filament-segmentation-2026')
print('downloaded to', path)

# locate the folder that contains train/test
base = path
if os.path.isdir(os.path.join(path, 'MAGFiLO_1.0_Kaggle_2026', 'train')):
    base = os.path.join(path, 'MAGFiLO_1.0_Kaggle_2026')
elif not os.path.isdir(os.path.join(path, 'train')):
    for d in os.listdir(path):
        if os.path.isdir(os.path.join(path, d, 'train')):
            base = os.path.join(path, d)
            break

os.environ['FILAMENT_BASE_PATH'] = base


In [ ]:
import os
os.makedirs(os.path.join(project_drive, 'code'), exist_ok=True)

In [ ]:
%%writefile /content/drive/MyDrive/filament_kaggle/code/config.py
# config.py - all the settings for the filament project
# change these for local vs colab vs kaggle runs

import os
import torch

# paths
# BASE_PATH is the folder that contains 'train' and 'test' subfolders.
# local default: data/MAGFiLO_1.0_Kaggle_2026
# kaggle notebook: /kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026
# colab: set FILAMENT_BASE_PATH to the downloaded dataset root

BASE_PATH = os.environ.get(
    "FILAMENT_BASE_PATH",
    r"C:\Users\srik2\Desktop\College\Machine Learning\Kaggle_Competition_Mini_Project\data\MAGFiLO_1.0_Kaggle_2026",
)

TRAIN_DIR = os.path.join(BASE_PATH, "train")
TEST_DIR = os.path.join(BASE_PATH, "test")
TRAIN_IMAGES = os.path.join(TRAIN_DIR, "train_images")
TEST_IMAGES = os.path.join(TEST_DIR, "test_images")
TRAIN_JSON = os.path.join(TRAIN_DIR, "MAGFiLO_1.0_Annotations_kaggle2026_train.json")

# where to save model, plots, submissions
PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
MODELS_DIR = os.path.join(PROJECT_ROOT, "code", "models")
PLOTS_DIR = os.path.join(PROJECT_ROOT, "plots")
SUBMISSIONS_DIR = os.path.join(PROJECT_ROOT, "submissions")

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(SUBMISSIONS_DIR, exist_ok=True)

# image constants
IMG_SIZE = 2048
TRAIN_RES = int(os.environ.get("FILAMENT_TRAIN_RES", "1024"))  # can be 512, 1024, 1536
INFERENCE_RES = int(os.environ.get("FILAMENT_INFERENCE_RES", str(IMG_SIZE)))  # usually 2048

# model
ENCODER = os.environ.get("FILAMENT_ENCODER", "tu-efficientnet_b3")
MODEL_NAME = "unet"
IN_CHANNELS = 1
CLASSES = 1

# training
SEED = int(os.environ.get("FILAMENT_SEED", "2026"))
BATCH_SIZE = int(os.environ.get("FILAMENT_BATCH_SIZE", "2"))
EPOCHS = int(os.environ.get("FILAMENT_EPOCHS", "50"))
LR = float(os.environ.get("FILAMENT_LR", "1e-3"))
MIN_LR = float(os.environ.get("FILAMENT_MIN_LR", "1e-5"))
WD = float(os.environ.get("FILAMENT_WD", "1e-4"))
PATIENCE = int(os.environ.get("FILAMENT_PATIENCE", "10"))
ACCUMULATION_STEPS = int(os.environ.get("FILAMENT_ACCUMULATION", "1"))

# loss weights
LOSS_DICE_W = float(os.environ.get("FILAMENT_LOSS_DICE_W", "0.4"))
LOSS_FOCAL_W = float(os.environ.get("FILAMENT_LOSS_FOCAL_W", "0.4"))
LOSS_BCE_W = float(os.environ.get("FILAMENT_LOSS_BCE_W", "0.2"))

# focal loss
FOCAL_GAMMA = 2.0
FOCAL_ALPHA = 0.25

# validation
N_FOLDS = int(os.environ.get("FILAMENT_N_FOLDS", "5"))
VAL_FOLD = int(os.environ.get("FILAMENT_VAL_FOLD", "0"))

# post-processing
PROB_THRESHOLD = float(os.environ.get("FILAMENT_PROB_THRESHOLD", "0.45"))
MORPH_CLOSE_K = int(os.environ.get("FILAMENT_MORPH_CLOSE_K", "3"))
MIN_AREA = int(os.environ.get("FILAMENT_MIN_AREA", "200"))
MAX_AREA = int(os.environ.get("FILAMENT_MAX_AREA", "500000"))

# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
%%writefile /content/drive/MyDrive/filament_kaggle/code/dataset.py
# dataset.py
# loads the COCO-style MAGFiLO annotations and creates a PyTorch Dataset
# for solar filament segmentation.

import json
import os
import random
from collections import defaultdict
from typing import List, Tuple

import cv2
import numpy as np
import torch
from torch.utils.data import Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2

# import config from the same folder
import config


def set_seed(seed: int = config.SEED):
    """fix randomness across the libraries we use."""
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def load_coco_annotations(json_path: str):
    """read the COCO json and return the whole dictionary."""
    with open(json_path, "r") as f:
        return json.load(f)


def group_annotations_by_filename(coco: dict, remove_ambiguous: bool = True):
    """
    the COCO json has 1154 image entries but only 707 actual jpg files.
    some files are annotated by multiple people, so they have multiple image_ids.
    this function groups all annotations by physical filename.

    returns:
        file_to_imgids: dict[filename] -> list of image_ids
        file_to_anns:   dict[filename] -> list of annotation dicts
    """
    filename_to_ids = defaultdict(list)
    for img in coco["images"]:
        filename_to_ids[img["file_name"]].append(img["id"])

    # collect annotations per image_id first
    id_to_anns = defaultdict(list)
    for ann in coco["annotations"]:
        if remove_ambiguous and ann.get("category_id") == 4:
            continue
        id_to_anns[ann["image_id"]].append(ann)

    file_to_imgids = dict(filename_to_ids)
    file_to_anns = {}
    for fname, imgids in file_to_imgids.items():
        anns = []
        for iid in imgids:
            anns.extend(id_to_anns[iid])
        file_to_anns[fname] = anns

    return file_to_imgids, file_to_anns


def polygon_to_mask(polygon: List[float], h: int, w: int) -> np.ndarray:
    """draw a closed polygon (flat list of x,y,x,y,...) as a binary mask."""
    pts = np.array(polygon, dtype=np.int32).reshape(-1, 2)
    # make sure the polygon is closed
    if not np.array_equal(pts[0], pts[-1]):
        pts = np.vstack([pts, pts[0]])
    mask = np.zeros((h, w), dtype=np.float32)
    cv2.fillPoly(mask, [pts], 1.0)
    return mask


def build_unified_mask(annotations: List[dict], h: int = 2048, w: int = 2048) -> np.ndarray:
    """
    combine every polygon in the annotation list into one binary mask.
    overlapping filaments all become 1 (semantic mask).
    """
    mask = np.zeros((h, w), dtype=np.float32)
    for ann in annotations:
        # segmentation is a list of one or more polygons
        segs = ann["segmentation"]
        if isinstance(segs, list):
            for seg in segs:
                if len(seg) >= 6:
                    m = polygon_to_mask(seg, h, w)
                    mask = np.maximum(mask, m)
    return mask


def get_train_val_files(file_to_imgids: dict, n_folds: int = 5, val_fold: int = 0, seed: int = 2026):
    """
    split the list of physical filenames into train and val.
    the split is by file, not by image_id, to avoid leakage.
    group = year from filename (positions 0-4) so all images of one year
    stay in the same fold.
    """
    from sklearn.model_selection import GroupKFold

    filenames = sorted(file_to_imgids.keys())
    groups = [fn[:4] for fn in filenames]

    gkf = GroupKFold(n_splits=n_folds)
    splits = list(gkf.split(filenames, [0] * len(filenames), groups))
    train_idx, val_idx = splits[val_fold]

    train_files = [filenames[i] for i in train_idx]
    val_files = [filenames[i] for i in val_idx]
    return train_files, val_files


def split_by_year(file_to_imgids: dict, train_years=range(2011, 2020), val_years=range(2020, 2023)):
    """alternative: hold out recent years as validation."""
    train_files = []
    val_files = []
    for fn in sorted(file_to_imgids.keys()):
        year = int(fn[:4])
        if year in train_years:
            train_files.append(fn)
        elif year in val_years:
            val_files.append(fn)
        else:
            # any other year goes to train
            train_files.append(fn)
    return train_files, val_files


def get_augmentations(image_size: int = config.TRAIN_RES, is_train: bool = True):
    """albumentations pipeline for training or validation."""
    if is_train:
        transform = A.Compose(
            [
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.5),
                A.RandomRotate90(p=0.5),
                A.Affine(
                    scale=(0.8, 1.2),
                    translate_percent=(-0.05, 0.05),
                    rotate=(-30, 30),
                    p=0.5,
                ),
                A.ElasticTransform(alpha=1, sigma=50, p=0.3),
                A.GridDistortion(p=0.3),
                A.RandomBrightnessContrast(
                    brightness_limit=0.2, contrast_limit=0.2, p=0.5
                ),
                A.GaussNoise(std_range=(0.01, 0.05), mean_range=(0.0, 0.0), p=0.3),
                A.GaussianBlur(blur_limit=3, p=0.3),
                A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.3),
                A.Normalize(mean=(0.5,), std=(0.5,)),
            ]
        )
    else:
        transform = A.Compose(
            [
                A.Resize(image_size, image_size),
                A.Normalize(mean=(0.5,), std=(0.5,)),
            ]
        )
    return transform


class FilamentDataset(Dataset):
    """
    PyTorch dataset for solar filament segmentation.

    for each physical jpg file we load the 2048x2048 image once,
    rasterize all annotations into a binary mask, then either
    random-crop a 1024x1024 region for training or resize to 1024 for validation.
    """

    def __init__(
        self,
        file_to_anns: dict,
        train_files: List[str],
        image_dir: str,
        image_size: int = config.TRAIN_RES,
        is_train: bool = True,
        seed: int = config.SEED,
    ):
        self.file_to_anns = file_to_anns
        self.image_dir = image_dir
        self.image_size = image_size
        self.is_train = is_train
        self.filenames = sorted(train_files)
        self.rng = np.random.default_rng(seed)

        # load all masks into memory once (707 images x 2048 x 2048 x 4 bytes ~ 11 GB, too much)
        # instead we only cache image id and annotation list, and load image on the fly.
        # if you have >16 GB ram you can cache the masks to speed training.
        self.transform = get_augmentations(image_size, is_train)

    def __len__(self):
        return len(self.filenames)

    def _load_image(self, fname: str) -> np.ndarray:
        path = os.path.join(self.image_dir, fname)
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise FileNotFoundError(f"image not found: {path}")
        return img.astype(np.float32) / 255.0

    def _load_mask(self, fname: str) -> np.ndarray:
        anns = self.file_to_anns[fname]
        if not anns:
            return np.zeros((2048, 2048), dtype=np.float32)
        return build_unified_mask(anns, 2048, 2048)

    def _sample_crop(self, img: np.ndarray, mask: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """
        for training, crop a square patch of size image_size from the 2048 image.
        with 75% probability the crop is centered on a random filament pixel,
        otherwise it is a random crop anywhere on the disk.
        """
        h, w = img.shape
        crop_h = crop_w = self.image_size

        # find foreground pixels
        ys, xs = np.where(mask > 0)
        if len(xs) > 0 and self.rng.random() < 0.75:
            idx = self.rng.integers(0, len(xs))
            cx, cy = xs[idx], ys[idx]
        else:
            cx = self.rng.integers(0, w)
            cy = self.rng.integers(0, h)

        # top-left corner
        x1 = min(max(cx - crop_w // 2, 0), w - crop_w)
        y1 = min(max(cy - crop_h // 2, 0), h - crop_h)

        return img[y1 : y1 + crop_h, x1 : x1 + crop_w], mask[y1 : y1 + crop_h, x1 : x1 + crop_w]

    def __getitem__(self, idx: int):
        fname = self.filenames[idx]
        img = self._load_image(fname)
        mask = self._load_mask(fname)

        if self.is_train:
            img, mask = self._sample_crop(img, mask)
        else:
            # for validation we resize to image_size, keeping aspect ratio not needed because square
            img = cv2.resize(img, (self.image_size, self.image_size), interpolation=cv2.INTER_AREA)
            mask = cv2.resize(
                mask, (self.image_size, self.image_size), interpolation=cv2.INTER_NEAREST
            )

        # albumentations
        sample = self.transform(image=img, mask=mask)
        img = sample["image"]  # (H, W)
        mask = sample["mask"]  # (H, W)

        # make tensors; add channel dim
        img_tensor = torch.from_numpy(img).unsqueeze(0).float()  # (1, H, W)
        mask_tensor = torch.from_numpy(mask).unsqueeze(0).float()  # (1, H, W)

        base_id = os.path.splitext(fname)[0]
        return img_tensor, mask_tensor, base_id


def make_dataloaders(
    coco_json_path: str = config.TRAIN_JSON,
    image_dir: str = config.TRAIN_IMAGES,
    n_folds: int = config.N_FOLDS,
    val_fold: int = config.VAL_FOLD,
    image_size: int = config.TRAIN_RES,
    batch_size: int = config.BATCH_SIZE,
    num_workers: int = 0,
    split_mode: str = "group_kfold",  # or "year"
):
    """build train and validation dataloaders."""
    coco = load_coco_annotations(coco_json_path)
    _, file_to_anns = group_annotations_by_filename(coco)

    if split_mode == "group_kfold":
        train_files, val_files = get_train_val_files(
            dict.fromkeys(file_to_anns.keys()), n_folds, val_fold
        )
    elif split_mode == "year":
        train_files, val_files = split_by_year(
            dict.fromkeys(file_to_anns.keys())
        )
    else:
        # random 85/15 by file
        all_files = sorted(file_to_anns.keys())
        from sklearn.model_selection import train_test_split
        train_files, val_files = train_test_split(
            all_files, test_size=0.15, random_state=config.SEED
        )

    train_ds = FilamentDataset(
        file_to_anns, train_files, image_dir, image_size, is_train=True
    )
    val_ds = FilamentDataset(
        file_to_anns, val_files, image_dir, image_size, is_train=False
    )

    train_loader = torch.utils.data.DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True if torch.cuda.is_available() else False,
        worker_init_fn=lambda x: set_seed(config.SEED + x),
    )
    val_loader = torch.utils.data.DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True if torch.cuda.is_available() else False,
    )

    print(f"train images: {len(train_ds)} | val images: {len(val_ds)}")
    return train_loader, val_loader, file_to_anns, val_files


if __name__ == "__main__":
    # quick sanity check
    set_seed()
    train_loader, val_loader, file_to_anns, val_files = make_dataloaders(num_workers=0)
    for batch in train_loader:
        img, mask, base_id = batch
        print("batch img shape:", img.shape, "mask shape:", mask.shape, "sample:", base_id[0])
        break


In [ ]:
%%writefile /content/drive/MyDrive/filament_kaggle/code/infer.py
# infer.py
# run the trained model over the test set (or validation) and build a submission.

import glob
import os

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from albumentations import Compose, Normalize
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm

import config
import model
import postprocess


def load_image_for_inference(path: str, image_size: int = config.TRAIN_RES) -> torch.Tensor:
    """
    load a 2048x2048 image and return a tensor (1, 1, image_size, image_size).
    we train at image_size, then resize the probability map to 2048.
    """
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"image not found: {path}")
    img = img.astype(np.float32) / 255.0
    # resize to image_size for the model
    img_small = cv2.resize(img, (image_size, image_size), interpolation=cv2.INTER_AREA)
    # normalize to mean 0.5 std 0.5
    img_small = (img_small - 0.5) / 0.5
    tensor = torch.from_numpy(img_small).unsqueeze(0).unsqueeze(0).float()  # (1, 1, H, W)
    return tensor, img.shape


def tta_inference(net, tensor, image_size, device, flips=["none", "h", "v"]):
    """
    simple test-time augmentation: horizontal and vertical flips.
    returns the average probability map at the model output resolution.
    """
    net.eval()
    avg = None
    for f in flips:
        x = tensor.clone().to(device)
        if f == "h":
            x = torch.flip(x, dims=[3])
        elif f == "v":
            x = torch.flip(x, dims=[2])
        elif f == "hv":
            x = torch.flip(x, dims=[2, 3])

        with torch.no_grad():
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
                logit = net(x)
                prob = torch.sigmoid(logit)

        # undo flips
        if f == "h":
            prob = torch.flip(prob, dims=[3])
        elif f == "v":
            prob = torch.flip(prob, dims=[2])
        elif f == "hv":
            prob = torch.flip(prob, dims=[2, 3])

        if avg is None:
            avg = prob
        else:
            avg += prob

    avg /= len(flips)
    return avg


def resize_prob_to_full(prob, full_shape=(2048, 2048)):
    """resize a probability map from train resolution to 2048."""
    prob = prob.squeeze(0).squeeze(0).cpu().numpy()
    full = cv2.resize(prob, (full_shape[1], full_shape[0]), interpolation=cv2.INTER_LINEAR)
    return full


def predict_image(
    net: nn.Module,
    path: str,
    device: torch.device,
    image_size: int = config.TRAIN_RES,
    use_tta: bool = True,
) -> np.ndarray:
    """return a 2048x2048 probability map for one image."""
    tensor, full_shape = load_image_for_inference(path, image_size)

    if use_tta:
        prob = tta_inference(net, tensor, image_size, device)
    else:
        net.eval()
        with torch.no_grad():
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
                logit = net(tensor.to(device))
        prob = torch.sigmoid(logit)

    full_prob = resize_prob_to_full(prob, full_shape)
    return full_prob


def run_on_directory(
    net: nn.Module,
    image_dir: str,
    output_csv: str,
    device: torch.device = config.device,
    image_size: int = config.TRAIN_RES,
    threshold: float = config.PROB_THRESHOLD,
    min_area: int = config.MIN_AREA,
    use_tta: bool = False,
):
    """generate a submission csv for all jpg files in image_dir."""
    rows = []
    image_files = sorted([
        p for p in glob.glob(os.path.join(image_dir, "*"))
        if os.path.splitext(p)[1].lower() in {".jpg", ".jpeg"}
    ])

    for path in tqdm(image_files, desc="infer"):
        fname = os.path.basename(path)
        base_id = os.path.splitext(fname)[0]

        full_prob = predict_image(net, path, device, image_size, use_tta)

        # mask the area outside the disk to avoid background false positives
        disk = postprocess.solar_disk_mask(cv2.imread(path, cv2.IMREAD_GRAYSCALE))

        rles, _ = postprocess.full_pipeline(
            full_prob,
            threshold=threshold,
            min_area=min_area,
            disk=disk,
        )

        for i, rle in enumerate(rles):
            rows.append({"filament_id": f"{base_id}_{i:04d}", "segmentation_rle": rle})

    df = pd.DataFrame(rows, columns=["filament_id", "segmentation_rle"])
    df.to_csv(output_csv, index=False)
    print(f"saved {output_csv} with {len(df)} filaments across {len(image_files)} images")
    return df


def run_validation_oof(
    fold: int = config.VAL_FOLD,
    image_size: int = config.TRAIN_RES,
    threshold: float = config.PROB_THRESHOLD,
    use_tta: bool = False,
):
    """out-of-fold inference on the validation files; useful for local PQ estimation."""
    from dataset import load_coco_annotations, group_annotations_by_filename, get_train_val_files, FilamentDataset
    import csv

    coco = load_coco_annotations(config.TRAIN_JSON)
    _, file_to_anns = group_annotations_by_filename(coco)
    train_files, val_files = get_train_val_files(
        dict.fromkeys(file_to_anns.keys()), config.N_FOLDS, fold
    )
    val_dir = config.TRAIN_IMAGES

    net = model.get_model().to(config.device)
    best_path = os.path.join(config.MODELS_DIR, f"best_fold_{fold}.pth")
    if os.path.exists(best_path):
        model.load_checkpoint(net, best_path)
    else:
        print(f"warning: no checkpoint found at {best_path}, using random weights")

    oof_csv = os.path.join(config.SUBMISSIONS_DIR, f"oof_fold_{fold}.csv")
    run_on_directory(net, val_dir, oof_csv, image_size=image_size, threshold=threshold, use_tta=use_tta)
    return oof_csv


def run_test_submission(
    checkpoint_path: str = None,
    output_csv: str = None,
    image_size: int = config.TRAIN_RES,
    threshold: float = config.PROB_THRESHOLD,
    use_tta: bool = False,
):
    """final submission on the 180 test images."""
    if checkpoint_path is None:
        # use the latest best checkpoint
        candidates = sorted(
            glob.glob(os.path.join(config.MODELS_DIR, "best_fold_*.pth"))
        )
        if not candidates:
            raise FileNotFoundError("no best_fold_*.pth found")
        checkpoint_path = candidates[-1]

    if output_csv is None:
        output_csv = os.path.join(
            config.SUBMISSIONS_DIR,
            f"submission_{os.path.basename(checkpoint_path).replace('.pth', '')}.csv",
        )

    net = model.get_model().to(config.device)
    model.load_checkpoint(net, checkpoint_path)

    return run_on_directory(
        net,
        config.TEST_IMAGES,
        output_csv,
        image_size=image_size,
        threshold=threshold,
        use_tta=use_tta,
    )


if __name__ == "__main__":
    # test with an untrained model (will be random)
    output = run_test_submission(
        image_size=512,
        threshold=0.5,
        use_tta=False,
    )
    print(output.head())


In [ ]:
%%writefile /content/drive/MyDrive/filament_kaggle/code/losses.py
# losses.py
# combined dice + focal + bce loss for filament segmentation.

import torch
import torch.nn as nn
import torch.nn.functional as F

import config


class DiceLoss(nn.Module):
    """soft dice loss for binary segmentation."""

    def __init__(self, smooth: float = 1e-6, from_logits: bool = True):
        super().__init__()
        self.smooth = smooth
        self.from_logits = from_logits

    def forward(self, y_pred: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
        if self.from_logits:
            y_pred = torch.sigmoid(y_pred)
        # flatten
        pred = y_pred.view(-1)
        true = y_true.view(-1)
        intersection = (pred * true).sum()
        union = pred.sum() + true.sum()
        dice = (2.0 * intersection + self.smooth) / (union + self.smooth)
        return 1.0 - dice


class FocalLoss(nn.Module):
    """binary focal loss for hard/misclassified pixels."""

    def __init__(
        self,
        alpha: float = config.FOCAL_ALPHA,
        gamma: float = config.FOCAL_GAMMA,
        from_logits: bool = True,
    ):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.from_logits = from_logits

    def forward(self, y_pred: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
        if self.from_logits:
            bce = F.binary_cross_entropy_with_logits(y_pred, y_true, reduction="none")
            p = torch.sigmoid(y_pred)
        else:
            bce = F.binary_cross_entropy(y_pred, y_true, reduction="none")
            p = y_pred

        # focal weight
        p_t = p * y_true + (1 - p) * (1 - y_true)
        weight = self.alpha * (1 - p_t) ** self.gamma
        loss = (weight * bce).mean()
        return loss


class BCELoss(nn.Module):
    """plain bce from logits."""

    def __init__(self, from_logits: bool = True):
        super().__init__()
        self.from_logits = from_logits

    def forward(self, y_pred: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
        if self.from_logits:
            return F.binary_cross_entropy_with_logits(y_pred, y_true)
        return F.binary_cross_entropy(y_pred, y_true)


class CombinedLoss(nn.Module):
    """
    weighted sum of dice, focal and bce.
    the official baseline says dice + focal, but the code uses dice + bce.
    we use all three because filament segmentation has class imbalance and
    thin boundaries.
    """

    def __init__(
        self,
        dice_w: float = config.LOSS_DICE_W,
        focal_w: float = config.LOSS_FOCAL_W,
        bce_w: float = config.LOSS_BCE_W,
    ):
        super().__init__()
        self.dice = DiceLoss()
        self.focal = FocalLoss()
        self.bce = BCELoss()
        self.dice_w = dice_w
        self.focal_w = focal_w
        self.bce_w = bce_w

    def forward(self, y_pred: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
        return (
            self.dice_w * self.dice(y_pred, y_true)
            + self.focal_w * self.focal(y_pred, y_true)
            + self.bce_w * self.bce(y_pred, y_true)
        )


if __name__ == "__main__":
    loss = CombinedLoss()
    pred = torch.randn(2, 1, 256, 256)
    true = torch.randint(0, 2, (2, 1, 256, 256)).float()
    l = loss(pred, true)
    print("combined loss:", l.item())


In [ ]:
%%writefile /content/drive/MyDrive/filament_kaggle/code/metrics.py
# metrics.py
# competition-style metrics: panoptic quality, dice, iou, hit/miss.

from typing import List, Tuple

import numpy as np
import pycocotools.mask as mask_util
import torch
import torch.nn.functional as F


def dice_score(pred: np.ndarray, true: np.ndarray, eps: float = 1e-7) -> float:
    """pixel-level dice for one binary mask."""
    pred = pred.astype(np.float32).ravel()
    true = true.astype(np.float32).ravel()
    inter = (pred * true).sum()
    return (2.0 * inter + eps) / (pred.sum() + true.sum() + eps)


def iou_score(pred: np.ndarray, true: np.ndarray, eps: float = 1e-7) -> float:
    """pixel-level intersection over union for one binary mask."""
    pred = pred.astype(np.float32).ravel()
    true = true.astype(np.float32).ravel()
    inter = (pred * true).sum()
    union = pred.sum() + true.sum() - inter
    return (inter + eps) / (union + eps)


def rles_to_layers(rles: List[str], height: int = 2048, width: int = 2048) -> np.ndarray:
    """
    decode a list of compressed COCO RLE strings into (n_masks, H, W) array.
    """
    if not rles:
        return np.zeros((0, height, width), dtype=np.float32)
    rle_dicts = [{"size": [height, width], "counts": rle} for rle in rles]
    masks = mask_util.decode(rle_dicts)
    return masks.transpose(2, 0, 1).astype(np.float32)


def get_overlap_matrices(
    gt_layers: torch.Tensor, pred_layers: torch.Tensor
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    pairwise iou and dice between GT and predicted instance masks.
    shapes: (n_gt, H, W) and (n_pred, H, W).
    """
    n_gt, h, w = gt_layers.shape
    n_pred = pred_layers.shape[0]

    gt_flat = gt_layers.reshape(n_gt, h * w)
    pred_flat = pred_layers.reshape(n_pred, h * w)

    intersection = torch.matmul(gt_flat, pred_flat.t())
    gt_areas = gt_flat.sum(dim=1, keepdim=True)
    pred_areas = pred_flat.sum(dim=1, keepdim=True).t()

    union = gt_areas + pred_areas - intersection
    iou = torch.where(
        union == 0, torch.tensor(0.0, device=gt_layers.device), intersection / union
    )
    dice = torch.where(
        union == 0,
        torch.tensor(0.0, device=gt_layers.device),
        2 * intersection / (gt_areas + pred_areas),
    )
    return iou, dice


def fp_count_hit(hit_matrix: torch.Tensor) -> int:
    """predicted masks that match no gt mask."""
    return (hit_matrix.sum(dim=0) == 0).sum().item()


def fn_count_hit(hit_matrix: torch.Tensor) -> int:
    """gt masks that match no predicted mask."""
    return (hit_matrix.sum(dim=1) == 0).sum().item()


def panoptic_quality(
    gt_layers: torch.Tensor,
    pred_layers: torch.Tensor,
    iou_threshold: float = 0.5,
) -> Tuple[float, int, int, int]:
    """
    compute PQ for one image.
    returns (pq_score, n_tp, n_fp, n_fn).
    """
    n_gt, n_pred = gt_layers.shape[0], pred_layers.shape[0]
    if n_gt == 0:
        return 0.0, 0, n_pred, 0
    if n_pred == 0:
        return 0.0, 0, 0, n_gt

    iou_matrix, _ = get_overlap_matrices(gt_layers, pred_layers)
    hit_matrix = iou_matrix > iou_threshold

    # each pair with iou > 0.5 counts as a true positive, summing its iou
    tp_iou_scores = iou_matrix[hit_matrix].tolist()
    tp = len(tp_iou_scores)
    fp = fp_count_hit(hit_matrix)
    fn = fn_count_hit(hit_matrix)

    denom = tp + 0.5 * fp + 0.5 * fn
    if denom == 0:
        return 0.0, tp, fp, fn
    pq = sum(tp_iou_scores) / denom
    return pq, tp, fp, fn


def mean_pq(pred_masks: List[np.ndarray], gt_masks: List[np.ndarray]) -> dict:
    """
    average PQ over a list of images.
    pred_masks and gt_masks are lists of (n_instances, H, W) arrays.
    """
    pq_scores = []
    total_tp = total_fp = total_fn = 0
    for p, g in zip(pred_masks, gt_masks):
        p = torch.from_numpy(p).float()
        g = torch.from_numpy(g).float()
        pq, tp, fp, fn = panoptic_quality(g, p)
        pq_scores.append(pq)
        total_tp += tp
        total_fp += fp
        total_fn += fn
    return {
        "mean_pq": float(np.mean(pq_scores)),
        "pq_scores": pq_scores,
        "total_tp": total_tp,
        "total_fp": total_fp,
        "total_fn": total_fn,
    }


def multiscale_iou(pred: np.ndarray, true: np.ndarray, scales: List[int] = [4, 8, 16, 32, 64]) -> float:
    """
    multi-scale IoU (MIoU): grid the image into overlapping windows of different
    sizes and average the IoU at each scale.
    """
    scores = []
    for s in scales:
        # stride half the window size
        stride = max(s // 2, 1)
        h, w = pred.shape
        local_scores = []
        for y in range(0, h - s + 1, stride):
            for x in range(0, w - s + 1, stride):
                p_patch = pred[y : y + s, x : x + s]
                t_patch = true[y : y + s, x : x + s]
                iou = iou_score(p_patch, t_patch)
                local_scores.append(iou)
        if local_scores:
            scores.append(np.mean(local_scores))
    return float(np.mean(scores)) if scores else 0.0


def calculate_batch_dice(pred: torch.Tensor, true: torch.Tensor) -> float:
    """simple batch dice (same as the public baseline name)."""
    pred = (torch.sigmoid(pred) > 0.5).float()
    pred = pred.detach().cpu().numpy()
    true = true.detach().cpu().numpy()
    return dice_score(pred, true)


if __name__ == "__main__":
    # tiny test
    g = torch.zeros((1, 1, 256, 256))
    g[0, 0, 50:80, 50:80] = 1
    p = g.clone()
    print("dice:", dice_score(p.numpy()[0, 0], g.numpy()[0, 0]))


In [ ]:
%%writefile /content/drive/MyDrive/filament_kaggle/code/model.py
# model.py
# builds the segmentation model using the segmentation_models_pytorch library.

import os
import segmentation_models_pytorch as smp
import torch
import torch.nn as nn

import config


def get_model(
    encoder_name: str = config.ENCODER,
    model_name: str = config.MODEL_NAME,
    in_channels: int = config.IN_CHANNELS,
    classes: int = config.CLASSES,
):
    """
    create a pretrained segmentation model.

    because the input is single channel (grayscale H-alpha), we set in_channels=1.
    smp will adapt the first conv layer for us.

    for a lightweight colab/T4 run we default to efficientnet-b3.
    if the encoder starts with 'tu-' it is from timm; otherwise it is a torch vision encoder.
    """
    model_name = model_name.lower()
    if model_name == "unet":
        model = smp.Unet(
            encoder_name=encoder_name,
            encoder_weights="imagenet",
            in_channels=in_channels,
            classes=classes,
            activation=None,
        )
    elif model_name == "unetplusplus":
        model = smp.UnetPlusPlus(
            encoder_name=encoder_name,
            encoder_weights="imagenet",
            in_channels=in_channels,
            classes=classes,
            activation=None,
        )
    elif model_name == "deeplabv3plus":
        model = smp.DeepLabV3Plus(
            encoder_name=encoder_name,
            encoder_weights="imagenet",
            in_channels=in_channels,
            classes=classes,
            activation=None,
        )
    elif model_name == "fpn":
        model = smp.FPN(
            encoder_name=encoder_name,
            encoder_weights="imagenet",
            in_channels=in_channels,
            classes=classes,
            activation=None,
        )
    else:
        raise ValueError(f"unknown model_name {model_name}")

    return model


def load_checkpoint(model, checkpoint_path: str):
    """load a saved state_dict into the model."""
    state = torch.load(checkpoint_path, map_location=config.device)
    if "model" in state:
        model.load_state_dict(state["model"])
    else:
        model.load_state_dict(state)
    print(f"loaded checkpoint from {checkpoint_path}")
    return model


if __name__ == "__main__":
    m = get_model()
    x = torch.randn(2, 1, config.TRAIN_RES, config.TRAIN_RES)
    y = m(x)
    print(f"input shape: {x.shape}  output shape: {y.shape}")


In [ ]:
%%writefile /content/drive/MyDrive/filament_kaggle/code/postprocess.py
# postprocess.py
# turn a probability map into a list of filament instance masks and encode them.

from typing import List, Tuple

import cv2
import numpy as np
from scipy import ndimage
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
import pycocotools.mask as mask_util


def solar_disk_mask(image: np.ndarray) -> np.ndarray:
    """
    estimate a binary solar disk mask for a 2048x2048 H-alpha image.
    uses a simple threshold + largest connected component.
    """
    if image.max() <= 1.0:
        img8 = (image * 255).astype(np.uint8)
    else:
        img8 = image.astype(np.uint8)

    # threshold well above the black padding
    _, binary = cv2.threshold(img8, 10, 255, cv2.THRESH_BINARY)
    # find largest component
    num, labels = cv2.connectedComponents(binary, connectivity=8)
    if num <= 1:
        return binary
    sizes = np.bincount(labels.ravel())
    sizes[0] = 0  # background
    largest = sizes.argmax()
    disk = (labels == largest).astype(np.uint8)
    return disk


def morphological_cleanup(mask: np.ndarray, close_ksize: int = 3) -> np.ndarray:
    """small closing to fill holes, then small opening to remove noise."""
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_ksize, close_ksize))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    return mask


def mask_to_instances(
    mask: np.ndarray,
    min_area: int = 200,
    max_area: int = 500000,
    watershed_split: bool = True,
) -> List[np.ndarray]:
    """
    split a binary mask into separate instance masks.

    - remove small components
    - optionally split merged components with watershed on distance transform
    - return a list of binary masks, one per filament
    """
    labeled, num = ndimage.label(mask)
    if num == 0:
        return []

    instances = []
    for i in range(1, num + 1):
        comp = (labeled == i).astype(np.uint8)
        area = comp.sum()
        if area < min_area or area > max_area:
            continue

        if watershed_split:
            # distance transform
            dist = ndimage.distance_transform_edt(comp)
            # find peaks that are not too close
            coords = peak_local_max(
                dist,
                min_distance=20,
                exclude_border=False,
                footprint=np.ones((5, 5)),
                labels=comp,
            )
            if len(coords) > 1:
                mask_internal = np.zeros(comp.shape, dtype=bool)
                mask_internal[tuple(coords.T)] = True
                markers, _ = ndimage.label(mask_internal)
                ws = watershed(-dist, markers, mask=comp.astype(bool), watershed_line=True)
                # extract the split components
                unique_markers = np.unique(markers)
                for m in unique_markers:
                    if m == 0:
                        continue
                    sub = (ws == m).astype(np.uint8)
                    if min_area <= sub.sum() <= max_area:
                        instances.append(sub)
                continue
        instances.append(comp)

    return instances


def rle_encode(mask: np.ndarray) -> str:
    """encode a single binary mask to COCO RLE with pycocotools."""
    return mask_util.encode(np.asfortranarray(mask.astype(np.uint8)))["counts"].decode("utf-8")


def rle_decode(rle: str, height: int = 2048, width: int = 2048) -> np.ndarray:
    """decode a COCO RLE string to binary mask."""
    return mask_util.decode({"size": [height, width], "counts": rle})


def threshold_search(
    model,
    loader,
    device,
    thresholds: np.ndarray = np.arange(0.3, 0.7, 0.05),
) -> float:
    """
    find the best probability threshold on a validation loader.
    we look for the threshold that maximizes PQ (or dice, if PQ is unavailable).
    """
    import metrics

    best_score = -1.0
    best_thr = 0.5
    for thr in thresholds:
        scores = []
        for imgs, masks, _ in loader:
            imgs = imgs.to(device)
            masks = masks.to(device)
            with torch.no_grad():
                logits = model(imgs)
                if logits.shape != masks.shape:
                    logits = torch.nn.functional.interpolate(
                        logits, size=masks.shape[2:], mode="bilinear", align_corners=False
                    )
                probs = torch.sigmoid(logits)
                preds = (probs > thr).float()

            for p, m in zip(preds.detach().cpu().numpy(), masks.detach().cpu().numpy()):
                pred_labeled, pred_n = ndimage.label(p[0])
                true_labeled, true_n = ndimage.label(m[0])
                pred_layers = np.stack([
                    (pred_labeled == i).astype(np.float32)
                    for i in range(1, pred_n + 1)
                ]) if pred_n > 0 else np.zeros((0, p.shape[1], p.shape[2]), dtype=np.float32)
                true_layers = np.stack([
                    (true_labeled == i).astype(np.float32)
                    for i in range(1, true_n + 1)
                ]) if true_n > 0 else np.zeros((0, m.shape[1], m.shape[2]), dtype=np.float32)

                pq, _, _, _ = metrics.panoptic_quality(
                    torch.from_numpy(true_layers), torch.from_numpy(pred_layers)
                )
                scores.append(pq)

        score = float(np.mean(scores))
        if score > best_score:
            best_score = score
            best_thr = thr
    print(f"best threshold {best_thr:.2f} with PQ {best_score:.4f}")
    return best_thr


def full_pipeline(
    prob: np.ndarray,
    threshold: float = 0.45,
    close_ksize: int = 3,
    min_area: int = 200,
    max_area: int = 500000,
    apply_watershed: bool = True,
    disk: np.ndarray = None,
) -> Tuple[List[str], List[np.ndarray]]:
    """
    take a (H, W) probability map and return:
        - list of RLE strings (one per filament)
        - list of binary masks
    """
    mask = (prob > threshold).astype(np.uint8)
    mask = morphological_cleanup(mask, close_ksize)

    if disk is not None:
        mask = mask * disk

    instances = mask_to_instances(mask, min_area, max_area, apply_watershed)
    rles = [rle_encode(inst) for inst in instances]
    return rles, instances


if __name__ == "__main__":
    # quick test
    fake = np.zeros((2048, 2048), dtype=np.float32)
    fake[500:600, 500:700] = 0.9
    fake[800:820, 800:1200] = 0.9
    rles, insts = full_pipeline(fake, threshold=0.5)
    print("encoded filaments:", len(rles))


In [ ]:
%%writefile /content/drive/MyDrive/filament_kaggle/code/submit.py
# submit.py
# simple command line wrapper to create a kaggle submission csv.

import argparse
import config
import infer


def main():
    parser = argparse.ArgumentParser(description="generate filament submission csv")
    parser.add_argument("--checkpoint", type=str, default=None, help="path to best_fold_*.pth")
    parser.add_argument("--out", type=str, default=None, help="output csv path")
    parser.add_argument("--image-size", type=int, default=config.TRAIN_RES)
    parser.add_argument("--threshold", type=float, default=config.PROB_THRESHOLD)
    parser.add_argument("--tta", action="store_true", help="use test-time augmentation")
    parser.add_argument("--val-fold", type=int, default=None, help="run on val set for OOF csv")
    args = parser.parse_args()

    if args.val_fold is not None:
        out = infer.run_validation_oof(
            fold=args.val_fold,
            image_size=args.image_size,
            threshold=args.threshold,
            use_tta=args.tta,
        )
    else:
        out = infer.run_test_submission(
            checkpoint_path=args.checkpoint,
            output_csv=args.out,
            image_size=args.image_size,
            threshold=args.threshold,
            use_tta=args.tta,
        )
    print("wrote", out)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/drive/MyDrive/filament_kaggle/code/train.py
# train.py
# full training loop for one fold.

import csv
import os
import time

import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import autocast, GradScaler
from tqdm import tqdm

import config
import dataset
import losses
import metrics
import model


def train_one_epoch(
    model: nn.Module,
    loader,
    criterion,
    optimizer,
    scheduler,
    scaler,
    accumulation_steps: int = config.ACCUMULATION_STEPS,
    device: torch.device = config.device,
):
    model.train()
    total_loss = 0.0
    total_dice = 0.0
    pbar = tqdm(loader, desc="train")
    optimizer.zero_grad()

    for step, (imgs, masks, _) in enumerate(pbar):
        imgs = imgs.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        with autocast(device_type=device.type, enabled=(device.type == "cuda")):
            logits = model(imgs)
            # resize logits if needed to match mask size
            if logits.shape != masks.shape:
                logits = nn.functional.interpolate(
                    logits, size=masks.shape[2:], mode="bilinear", align_corners=False
                )
            loss = criterion(logits, masks)

        if accumulation_steps > 1:
            loss = loss / accumulation_steps

        scaler.scale(loss).backward()

        if (step + 1) % accumulation_steps == 0 or (step + 1) == len(loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        with torch.no_grad():
            dice = metrics.dice_score(
                (torch.sigmoid(logits) > 0.5).float().detach().cpu().numpy(),
                masks.detach().cpu().numpy(),
            )

        total_loss += loss.item() * accumulation_steps
        total_dice += dice
        pbar.set_postfix({"loss": f"{total_loss / (step + 1):.4f}", "dice": f"{total_dice / (step + 1):.4f}"})

    return total_loss / len(loader), total_dice / len(loader)


@torch.no_grad()
def validate(
    model: nn.Module,
    loader,
    criterion,
    device: torch.device = config.device,
    threshold: float = config.PROB_THRESHOLD,
):
    """
    validation that also computes panoptic quality on the predicted instances.
    """
    model.eval()
    total_loss = 0.0
    total_dice = 0.0
    all_pq = []

    pbar = tqdm(loader, desc="val")
    for imgs, masks, _ in pbar:
        imgs = imgs.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        with autocast(device_type=device.type, enabled=(device.type == "cuda")):
            logits = model(imgs)
            if logits.shape != masks.shape:
                logits = nn.functional.interpolate(
                    logits, size=masks.shape[2:], mode="bilinear", align_corners=False
                )
            loss = criterion(logits, masks)

        probs = torch.sigmoid(logits)
        preds = (probs > threshold).float()

        total_loss += loss.item()
        # dice is computed at current resolution (1024 or 512)
        for p, m in zip(preds.detach().cpu().numpy(), masks.detach().cpu().numpy()):
            dice = metrics.dice_score(p[0], m[0])
            total_dice += dice

        # for PQ we need instance masks; treat each connected component of the
        # prediction as an instance. because the masks are at train resolution we
        # get a lower-bound PQ, but it is still useful for model selection.
        for p, m in zip(preds.detach().cpu().numpy(), masks.detach().cpu().numpy()):
            # connected components on predicted and gt masks
            from scipy import ndimage
            pred_labeled, pred_n = ndimage.label(p[0])
            true_labeled, true_n = ndimage.label(m[0])

            pred_layers = np.zeros((pred_n, p.shape[1], p.shape[2]), dtype=np.float32)
            for i in range(1, pred_n + 1):
                pred_layers[i - 1] = (pred_labeled == i).astype(np.float32)

            true_layers = np.zeros((true_n, m.shape[1], m.shape[2]), dtype=np.float32)
            for i in range(1, true_n + 1):
                true_layers[i - 1] = (true_labeled == i).astype(np.float32)

            pq, _, _, _ = metrics.panoptic_quality(
                torch.from_numpy(true_layers), torch.from_numpy(pred_layers)
            )
            all_pq.append(pq)

    mean_pq = float(np.mean(all_pq)) if all_pq else 0.0
    return (
        total_loss / len(loader),
        total_dice / len(loader),
        mean_pq,
    )


def train_fold(
    fold: int = config.VAL_FOLD,
    epochs: int = config.EPOCHS,
    save_dir: str = config.MODELS_DIR,
    log_dir: str = None,
):
    """train one fold and save the best checkpoint by validation PQ."""
    if log_dir is None:
        log_dir = save_dir
    print("training on device:", config.device)
    print(f"fold {fold} | {epochs} epochs | resolution {config.TRAIN_RES}")

    dataset.set_seed(config.SEED + fold)

    train_loader, val_loader, _, _ = dataset.make_dataloaders(
        val_fold=fold,
        image_size=config.TRAIN_RES,
        batch_size=config.BATCH_SIZE,
        num_workers=0,  # set to 2+ on colab/kaggle
    )

    net = model.get_model().to(config.device)

    criterion = losses.CombinedLoss()
    optimizer = AdamW(
        net.parameters(), lr=config.LR, weight_decay=config.WD, amsgrad=False
    )
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=config.MIN_LR)
    scaler = GradScaler(enabled=(config.device.type == "cuda"))

    csv_path = os.path.join(save_dir, f"fold_{fold}_metrics.csv")
    with open(csv_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["epoch", "train_loss", "train_dice", "val_loss", "val_dice", "val_pq", "best"])

    best_pq = -1.0
    best_path = os.path.join(save_dir, f"best_fold_{fold}.pth")

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        train_loss, train_dice = train_one_epoch(
            net, train_loader, criterion, optimizer, scheduler, scaler
        )
        val_loss, val_dice, val_pq = validate(net, val_loader, criterion)
        scheduler.step()

        is_best = val_pq > best_pq
        if is_best:
            best_pq = val_pq
            torch.save({"model": net.state_dict(), "pq": best_pq}, best_path)

        epoch_time = time.time() - t0
        print(
            f"epoch {epoch}/{epochs} | train loss {train_loss:.4f} dice {train_dice:.4f} | "
            f"val loss {val_loss:.4f} dice {val_dice:.4f} pq {val_pq:.4f} | "
            f"best pq {best_pq:.4f} | time {epoch_time:.1f}s"
        )

        with open(csv_path, "a", newline="") as f:
            writer = csv.writer(f)
            writer.writerow([epoch, f"{train_loss:.6f}", f"{train_dice:.4f}", f"{val_loss:.6f}", f"{val_dice:.4f}", f"{val_pq:.4f}", int(is_best)])

    print(f"fold {fold} finished. best pq {best_pq:.4f} saved to {best_path}")
    return best_path


if __name__ == "__main__":
    train_fold()


In [ ]:
import sys, os
sys.path.append(os.path.join(project_drive, 'code'))

os.environ['FILAMENT_TRAIN_RES'] = '1024'
os.environ['FILAMENT_INFERENCE_RES'] = '2048'
os.environ['FILAMENT_ENCODER'] = 'tu-efficientnet_b3'
os.environ['FILAMENT_BATCH_SIZE'] = '1'
os.environ['FILAMENT_ACCUMULATION'] = '2'
os.environ['FILAMENT_EPOCHS'] = '30'
os.environ['FILAMENT_LR'] = '2e-4'
os.environ['FILAMENT_MIN_LR'] = '1e-6'
os.environ['FILAMENT_PATIENCE'] = '10'
os.environ['FILAMENT_VAL_FOLD'] = '0'

import config, train, infer

best = train.train_fold(fold=0, epochs=config.EPOCHS, save_dir=project_drive)

sub = infer.run_test_submission(
    checkpoint_path=best,
    output_csv=os.path.join(project_drive, 'submission.csv'),
    image_size=config.TRAIN_RES,
    threshold=0.45,
    use_tta=True,
)
print(sub.head())


In [ ]:
import pandas as pd
sub = pd.read_csv(os.path.join(project_drive, 'submission.csv'))
print(sub.shape)
print(sub.head())
assert sub.columns.tolist() == ['filament_id', 'segmentation_rle']
